## IMPORTS

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split

## DEFINITION OF VARIABLES

In [ ]:
SOURCE_POLYMER_FILE = 'all_spectra.csv'
SOURCE_MICROPLASTIC_FILE = 'ambient_spectra.csv'

## Affine Coupling Layer (Conv1D)

In [ ]:
import torch
import torch.nn as nn

class ConvSubNet1D(nn.Module):
    def __init__(self, in_dim, out_dim, hidden_channels=32, kernel_size=5):
        super().__init__()
        padding = kernel_size // 2

        self.net = nn.Sequential(
            nn.Conv1d(1, hidden_channels, kernel_size=kernel_size, padding=padding),
            nn.ReLU(),
            nn.Conv1d(hidden_channels, hidden_channels, kernel_size=kernel_size, padding=padding),
            nn.ReLU(),
            nn.Conv1d(hidden_channels, hidden_channels, kernel_size=kernel_size, padding=padding),
            nn.ReLU(),
            nn.Conv1d(hidden_channels, 1, kernel_size=kernel_size, padding=padding)
        )

        self.in_dim = in_dim
        self.out_dim = out_dim

    def forward(self, x):
        # x: [batch, in_dim]
        x = x.unsqueeze(1)          # [batch, 1, in_dim]
        out = self.net(x)           # [batch, 1, in_dim]
        out = out.squeeze(1)        # [batch, in_dim]
        if out.shape[1] < self.out_dim:
            out = F.pad(out, (0, self.out_dim - out.shape[1]))
        elif out.shape[1] > self.out_dim:
            out = out[:, :self.out_dim]
        return out


class AffineCouplingLayer(nn.Module):
    def __init__(self, dim, hidden_channels=32, kernel_size=5, scale_factor=0.3):
        super().__init__()
        self.dim = dim
        self.split_dim = (dim + 1) // 2
        self.transform_dim = dim - self.split_dim
        self.scale_factor = scale_factor

        self.scale_net = ConvSubNet1D(
            in_dim=self.split_dim,
            out_dim=self.transform_dim,
            hidden_channels=hidden_channels,
            kernel_size=kernel_size
        )

        self.shift_net = ConvSubNet1D(
            in_dim=self.split_dim,
            out_dim=self.transform_dim,
            hidden_channels=hidden_channels,
            kernel_size=kernel_size
        )

    def forward(self, x):
        x1, x2 = x[:, :self.split_dim], x[:, self.split_dim:]

        s = self.scale_net(x1)
        s = torch.tanh(s) * self.scale_factor
        t = self.shift_net(x1)

        y1 = x1
        y2 = x2 * torch.exp(s) + t

        log_det = s.sum(dim=1)

        return torch.cat([y1, y2], dim=1), log_det

    def inverse(self, y):
        y1, y2 = y[:, :self.split_dim], y[:, self.split_dim:]

        s = self.scale_net(y1)
        s = torch.tanh(s) * self.scale_factor
        t = self.shift_net(y1)

        x1 = y1
        x2 = (y2 - t) * torch.exp(-s)

        return torch.cat([x1, x2], dim=1)

In [ ]:
class StandardGaussian:
    def __init__(self, dim):
        self.dim = dim
        self.distribution = torch.distributions.MultivariateNormal(
            torch.zeros(dim),
            torch.eye(dim)
        )

    def log_prob(self, z):
        return self.distribution.log_prob(z)
    
    def sample(self, n_samples):
        return self.distribution.sample((n_samples,))

In [ ]:
class PermutationLayer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.register_buffer('permutation', torch.randperm(dim))
        self.register_buffer('inverse_permutation', torch.argsort(self.permutation))

    def forward(self, x):
        return x[:, self.permutation], torch.zeros(x.shape[0], device=x.device)
    
    def inverse(self, y):
        return y[:,self.inverse_permutation]

In [ ]:
class NormalizingFlow(nn.Module):
    def __init__(self, dim, n_layers = 4, hidden_channels = 32):
        super().__init__()
        self.dim = dim
        self.n_layers = n_layers
        self.base_dist = StandardGaussian(dim)
        self.layers = nn.ModuleList()
        for i in range(n_layers):
            self.layers.append(AffineCouplingLayer(dim, hidden_channels=hidden_channels))
            if i < n_layers - 1:
                self.layers.append(PermutationLayer(dim))

    def forward(self, x):

        z = x
        log_det_sum = torch.zeros(x.shape[0], device=x.device)

        for layer in self.layers:
            z, log_det = layer(z)
            log_det_sum += log_det

        log_prob_z = self.base_dist.log_prob(z)
        log_prob_x = log_prob_z + log_det_sum

        return log_prob_x
    
    def inverse(self, z):
        x = z
        for layer in reversed(self.layers):
            x = layer.inverse(x)
        return x
    
    def sample(self, n_samples):
        z = self.base_dist.sample(n_samples)
        x = self.inverse(z)
        return x

In [ ]:
def train_flow(model, data, n_epochs = 100, batch_size = 32, lr = 1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr = lr)

    n_batches = len(data) // batch_size
    losses = []

    for epoch in tqdm(range(n_epochs)):
        perm = torch.randperm(len(data))
        data_shuffled = data[perm]

        epoch_loss = 0

        for i in range(n_batches):
            batch = data_shuffled[i*batch_size:(i+1)*batch_size]
            
            optimizer.zero_grad()

            log_prob = model(batch)
            loss = -log_prob.mean()

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / n_batches
        losses.append(avg_loss)

        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1}/{n_epochs}, Loss: {avg_loss:.4f}')

    return losses

## Training

In [ ]:
# Load data
df_polymer = pd.read_csv(SOURCE_POLYMER_FILE, index_col=False)
df_amb = pd.read_csv(SOURCE_MICROPLASTIC_FILE, index_col=False)
# Dropping singular samples from the polymer dataset to ensure stratified splitting works correctly
df_polymer = df_polymer[df_polymer['polymer'].map(df_polymer['polymer'].value_counts()) > 1]
df_train, df_polymer_test = train_test_split(df_polymer, test_size = 0.2, stratify = df_polymer['polymer'], random_state = 42)
df_test = pd.concat([df_polymer_test, df_amb], ignore_index=True)
feature_cols = [
    col for col in df_train.columns
    if col != 'polymer' and str(col) != '' and not str(col).startswith('Unnamed')
]
print(f"Training data (normal polymers): {df_train.shape[0]} samples")
print(f"Test data (abnormal microplastics): {df_test.shape[0]} samples")

In [ ]:
df_test['polymer'].value_counts()

In [ ]:
# Prepare training data (normal)
normal_data = df_train[feature_cols].values

# ALTERNATIVE: Use RobustScaler if values are too extreme
scaler = StandardScaler()
normal_scaled = scaler.fit_transform(normal_data)
train_tensor = torch.FloatTensor(normal_scaled)

# Initialize and train model
n_features = train_tensor.shape[1]
model = NormalizingFlow(dim = n_features, n_layers = 4, hidden_channels = 32)

print(f"\nTraining Normalizing Flow model (WITH STANDARD SCALER, NO SVD)...")
print(f"Using {n_features} spectral features after dropping metadata/index columns")
print(f"Feature 0 range: [{normal_data[:, 0].min():.2f}, {normal_data[:, 0].max():.2f}]")
losses = train_flow(model, train_tensor, n_epochs = 20, batch_size = 64, lr = 1e-3)

In [ ]:
# Plot training loss to diagnose convergence
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(losses, linewidth=2, color='blue')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Training Loss Curve', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final loss: {losses[-1]:.4f}")
print(f"Loss change (first to last epoch): {losses[0] - losses[-1]:.4f}")
if losses[-1] > losses[-10]:
    print("\u26a0\ufe0f  WARNING: Loss is increasing in recent epochs - model may not be converging well")
else:
    print("\u2713 Loss is decreasing - training appears stable")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score

# ============================================================================
# EVALUATION: Classify test data as anomaly or normal
# Test set contains:
#   - df_polymer_test samples (NORMAL - known polymers held out from training)
#   - df_amb samples (ANOMALY - ambient microplastics)
# ============================================================================

model.eval()

# Step 1: Score TRAINING data to compute threshold
with torch.no_grad():
    train_log_probs = model(train_tensor).cpu().numpy()

# Compute threshold from training data statistics
THRESHOLD = 25
train_mean = train_log_probs.mean()
train_std = train_log_probs.std()
threshold_log_prob = np.percentile(train_log_probs, THRESHOLD)
threshold_anomaly_score = -threshold_log_prob

print(f"{'='*70}")
print(f"ANOMALY DETECTION: Training Data Statistics")
print(f"{'='*70}")
print(f"Training samples: {len(train_log_probs)}")
print(f"Mean log probability: {train_mean:.4f}")
print(f"Std log probability: {train_std:.4f}")
print(f"Threshold ({THRESHOLD}th percentile): {threshold_log_prob:.4f}")
print(f"Threshold (anomaly score): {threshold_anomaly_score:.4f}")

# Step 2: Score TEST data (NO SVD — direct scaling only)
print(f"\n{'='*70}")
print(f"CLASSIFYING TEST DATA")
print(f"{'='*70}\n")

with torch.no_grad():
    test_data = df_test[feature_cols].values
    if scaler is not None:
        test_scaled = scaler.transform(test_data)
    else:
        test_scaled = test_data.astype(np.float32)
    test_tensor = torch.FloatTensor(test_scaled)
    test_log_probs = model(test_tensor).cpu().numpy()

test_anomaly_scores = -test_log_probs
test_predictions = (test_anomaly_scores >= threshold_anomaly_score).astype(int)

# Step 3: Create ground truth labels
# Polymer test samples are NORMAL (0), ambient samples are ANOMALY (1)
n_polymer_test = len(df_polymer_test)
n_amb = len(df_amb)
ground_truth = np.array([0] * n_polymer_test + [1] * n_amb)

# Step 4: Create results dataframe
results_df = pd.DataFrame({
    'Sample_Index': range(len(test_log_probs)),
    'Log_Probability': test_log_probs.flatten(),
    'Anomaly_Score': test_anomaly_scores.flatten(),
    'Ground_Truth': ground_truth,
    'Predicted': test_predictions,
    'Ground_Truth_Label': ['ANOMALY' if g == 1 else 'NORMAL' for g in ground_truth],
    'Predicted_Label': ['ANOMALY' if p == 1 else 'NORMAL' for p in test_predictions]
})

if 'polymer' in df_test.columns:
    results_df['Polymer_Type'] = df_test['polymer'].values

# Display summary
anomaly_count = test_predictions.sum()
normal_count = len(test_predictions) - anomaly_count

print(f"Total test samples: {len(test_predictions)}")
print(f"  - Normal polymer samples (expected NORMAL): {n_polymer_test}")
print(f"  - Ambient microplastic samples (expected ANOMALY): {n_amb}")
print(f"\nPredicted as ANOMALY: {anomaly_count} ({100*anomaly_count/len(test_predictions):.1f}%)")
print(f"Predicted as NORMAL: {normal_count} ({100*normal_count/len(test_predictions):.1f}%)")

# Step 5: Compute proper metrics
print(f"\n{'='*70}")
print(f"CLASSIFICATION METRICS")
print(f"{'='*70}\n")

precision = precision_score(ground_truth, test_predictions, zero_division=0)
recall = recall_score(ground_truth, test_predictions, zero_division=0)
f1 = f1_score(ground_truth, test_predictions, zero_division=0)

print(f"Precision: {precision:.4f}  (of predicted anomalies, how many are truly anomalous)")
print(f"Recall:    {recall:.4f}  (of actual anomalies, how many were detected)")
print(f"F1 Score:  {f1:.4f}")

print(f"\nFull Classification Report:")
print(classification_report(ground_truth, test_predictions, target_names=['NORMAL', 'ANOMALY']))

cm = confusion_matrix(ground_truth, test_predictions)
print(f"Confusion Matrix:")
print(f"                  Predicted NORMAL  Predicted ANOMALY")
print(f"  Actual NORMAL   {cm[0,0]:>16}  {cm[0,1]:>17}")
print(f"  Actual ANOMALY  {cm[1,0]:>16}  {cm[1,1]:>17}")

# Step 6: Per-sample results
print(f"\n{'='*70}")
print(f"PER-SAMPLE RESULTS (first 20 samples)")
print(f"{'='*70}\n")
print(results_df.head(20).to_string(index=False))

if len(results_df) > 20:
    print(f"\n... and {len(results_df) - 20} more samples")

# Step 7: Per-polymer breakdown
if 'polymer' in df_test.columns:
    print(f"\n{'='*70}")
    print(f"RESULTS BY POLYMER TYPE")
    print(f"{'='*70}")
    
    for polymer in df_test['polymer'].unique():
        polymer_mask = results_df['Polymer_Type'] == polymer
        polymer_results = results_df[polymer_mask]
        
        anomaly_detected = polymer_results['Predicted'].sum()
        total = len(polymer_results)
        pct = 100 * anomaly_detected / total
        avg_anomaly_score = polymer_results['Anomaly_Score'].mean()
        expected = 'ANOMALY' if polymer_results['Ground_Truth'].iloc[0] == 1 else 'NORMAL'
        
        print(f"\n{polymer} (expected: {expected}):")
        print(f"  Total samples: {total}")
        print(f"  Detected as ANOMALY: {anomaly_detected}/{total} ({pct:.1f}%)")
        print(f"  Avg anomaly score: {avg_anomaly_score:.4f}")

# Step 8: Visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Histogram of log probabilities
ax = axes[0, 0]
ax.hist(train_log_probs, bins=50, alpha=0.6, label='Training (Normal)', 
        color='green', edgecolor='black', density=True)
# Separate test into polymer_test (normal) and ambient (anomaly)
polymer_test_log_probs = test_log_probs[:n_polymer_test]
amb_log_probs = test_log_probs[n_polymer_test:]
ax.hist(polymer_test_log_probs, bins=30, alpha=0.5, label='Test Polymers (Normal)', 
        color='blue', edgecolor='black', density=True)
ax.hist(amb_log_probs, bins=30, alpha=0.5, label='Test Ambient (Anomaly)', 
        color='red', edgecolor='black', density=True)
ax.axvline(threshold_log_prob, color='black', linestyle='--', linewidth=2, 
          label=f'Threshold = {threshold_log_prob:.2f}')
ax.set_xlabel('Log Probability', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Log Probability Distributions', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Plot 2: Confusion matrix heatmap
ax = axes[0, 1]
sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn_r', 
            xticklabels=['NORMAL', 'ANOMALY'], yticklabels=['NORMAL', 'ANOMALY'], ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title(f'Confusion Matrix (F1={f1:.3f})', fontsize=14, fontweight='bold')

# Plot 3: Anomaly scores scatter plot colored by ground truth
ax = axes[1, 0]
normal_mask = ground_truth == 0
anomaly_mask = ground_truth == 1
ax.scatter(np.where(normal_mask)[0], test_anomaly_scores[normal_mask], 
           c='green', alpha=0.6, s=30, label='Normal (polymer test)')
ax.scatter(np.where(anomaly_mask)[0], test_anomaly_scores[anomaly_mask], 
           c='red', alpha=0.6, s=30, label='Anomaly (ambient)')
ax.axhline(threshold_anomaly_score, color='black', linestyle='--', linewidth=2, label='Threshold')
ax.set_xlabel('Sample Index', fontsize=12)
ax.set_ylabel('Anomaly Score', fontsize=12)
ax.set_title('Anomaly Scores by Ground Truth', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Plot 4: Detection rates by polymer type
if 'polymer' in df_test.columns:
    ax = axes[1, 1]
    polymer_types = results_df['Polymer_Type'].unique()
    detection_rates = []
    labels_by_polymer = []
    bar_colors = []
    
    for polymer in polymer_types:
        polymer_mask = results_df['Polymer_Type'] == polymer
        polymer_results = results_df[polymer_mask]
        total = len(polymer_results)
        anomaly_detected = polymer_results['Predicted'].sum()
        detection_rates.append(100 * anomaly_detected / total)
        expected = 'A' if polymer_results['Ground_Truth'].iloc[0] == 1 else 'N'
        labels_by_polymer.append(f"{polymer}\n({total} samp, exp:{expected})")
        bar_colors.append('red' if expected == 'A' else 'green')
    
    bars = ax.bar(range(len(polymer_types)), detection_rates, color=bar_colors, alpha=0.7, edgecolor='black', linewidth=2)
    ax.set_xticks(range(len(polymer_types)))
    ax.set_xticklabels(labels_by_polymer, fontsize=8)
    ax.set_ylabel('% Detected as Anomaly', fontsize=12)
    ax.set_title('Anomaly Detection Rate by Polymer Type', fontsize=14, fontweight='bold')
    ax.grid(alpha=0.3, axis='y')
    for bar, rate in zip(bars, detection_rates):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height, f'{rate:.1f}%',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
else:
    axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

# Step 9: Save results to CSV
output_filename = 'anomaly_detection_results_conv_no_svd.csv'
results_df.to_csv(output_filename, index=False)

print(f"\nResults saved to: {output_filename}")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Compute all metrics
accuracy = accuracy_score(ground_truth, test_predictions)
precision = precision_score(ground_truth, test_predictions, zero_division=0)
recall = recall_score(ground_truth, test_predictions, zero_division=0)
f1 = f1_score(ground_truth, test_predictions, zero_division=0)
# AUROC uses continuous anomaly scores (not binary predictions)
auroc = roc_auc_score(ground_truth, test_anomaly_scores)

metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUROC'],
    'Value': [accuracy, precision, recall, f1, auroc]
})
metrics_df['Value'] = metrics_df['Value'].map(lambda x: f"{x:.4f}")

print(f"{'='*40}")
print(f"  CLASSIFICATION METRICS SUMMARY")
print(f"  (Conv1D, No SVD)")
print(f"{'='*40}")
print(metrics_df.to_string(index=False))
print(f"{'='*40}")
print(f"\nThreshold: {THRESHOLD}th percentile of training log-probs")